# Large Language Model (LLM) Overview

## Let's get started!

We're going to start very quickly with something that should be familiar to you--prompting--to see how we can do the same thing we currently do in a web browser, instead, using the `pipeline` functions included in HuggingFace's `transformers` library. As discussed in the mini-lecture, `transformers` gives us some objects and functions that we can use to access LLMs comptuationally.

We also need to import a library called `torch` which gives us a data structure called a Tensor which is, essentially, a multidimensional array.

In [ ]:
# import libraries
from transformers import pipeline
import torch

### Picking the model

Now we pick the model. You (maybe?) already know the names of some of the biggies--GPT-4o, GPT-5, and the like. But as it turns out, there are SO MANY models. Like, tens and tens of thousands. [Here is a good overview](https://huggingface.co/docs/hub/en/transformers) of the various flavors, as well as instructions for how you can tell which ones you can access via `transformers`.


### Our first model

Let's just start with [GPT-2](https://huggingface.co/openai-community/gpt2), the model that Hugging Face suggests on its tutorial page. It's an older GPT, which is why it's now open and free to use. While its output is not quite as nuanced as more recent GPTs, it's also smaller and therefore faster to query, and requires fewer resources (both compute and energy) to use.

This is how you specify the model and set it up for a `text-generation` pipeline (task):

In [ ]:
# for nicely formatted printing
import textwrap

# torch.manual_seed(0)   # uncomment for reproducibility
generator = pipeline('text-generation', # this is the task we tell it to do
                     model = 'openai-community/gpt2',  # this is where we set the model
                     # device = 'cuda' # set cuda for GPUs; we don't need this yet so let's not
                     )

prompt = "People have a lot of opinions about cilantro. Some people think cilantro tastes like"

responses = generator(prompt,
                      max_new_tokens = 100,
                      num_return_sequences=5,
                      temperature=1.0,
                      truncation=True
                      )

#  print out the responses
for i, response in enumerate(responses):
    print()
    print(str(i) + ". " + textwrap.fill(response['generated_text'],100))


## A quick aside about temperature

Any parameter that you can set outside of a model is called a hyperparameter. (Fancy!) With respect to LLMs, there also aren't a lot. But the most interesting one to play with is called temperature, which impacts the randomness of the model's output.

First off, why is it called temperature? Tl;dr: physics!

Similar to how actual temperature (like of the air) influences the movement of particles in a system, with higher temperatures leading to more unpredictable behavior and lower temperatures resulting in more predictable, deterministic outputs, so too does the teperature parameter.

Essentially, it controls the "heat" of the model's decision-making process when choosing the next word to generate.

## Let's try it out!

To get a more intuitive understanding of this, let's imagine that the model only has five tokens in its vocabulary (instead of 50,000+). A schematic illustration of those probabilities might look like this:

    prompt: Whose woods these are I think I
    probabilities:
        know -> 0.5
        knew -> 0.2
        smell -> 0.15
        see -> 0.1
        am -> 0.05

The probabilities will add up to 1.0. A probability of 0.5 indicates that the token has a 50% probability of coming next; a probability of 0.2 means that the token has a 20% probability of coming next, etc.

Here are those probabilities represented in Python as two lists—one with the words, and one with the probabilities that correspond to those words by index:

In [ ]:
tokens = ['know', 'knew', 'smell', 'see', 'am']
probs = [0.5, 0.2, 0.15, 0.1, 0.05]

By default, to select the next token, the generation code picks from this list weighted by probability. The code to do this with PyTorch looks like this:

In [ ]:
index = torch.multinomial(torch.tensor(probs), 1).item()
print(tokens[index])

You don't have to worry about the specifics of this code—I'm just using it to demonstrate how the sampling process works. Run the code a few times and you'll see that about half the time you get "knew"—the token with the highest probability. Running the code in a loop makes this a bit easier to see:

In [ ]:
for i in range(10):
    index = torch.multinomial(torch.tensor(probs), 1).item()
    print(tokens[index])

Enter temperature. This lets you shift the probability distribution of the next token before it's sampled. It's, in effect, a multiplier.

So, if the temperature parameter is 1.0, then sampling will proceed as normal, with the tokens weighted by their estimated probability. If the temperature parameter is less than 1.0, then tokens that were already probable will get more probable. If the temperature parameter is greater than 1.0, then the probabilities start to even out, approaching a uniform distribution (meaning that no token is more likely to be chosen than any other).

To demonstrate this, I've written some code below that applies temperature to the probabilities defined above, and shows the resulting changes:

In [ ]:
for temperature in [0.1, 0.35, 1.0, 2.0, 50.0]:
    modified = torch.softmax(
        torch.log(torch.tensor(probs)) / temperature, dim=-1)

    print(f"temperature {temperature:0.02f}")

    for tok, prob in zip(tokens, modified):
        print(tok.ljust(6), "→", f"{prob:0.002f}")

    print()

You can see that at temperature 1.0, the probabilities are identical to the original. At temperature 0.35, the probability of the most likely token has been boosted, but the other tokens still have a small chance of occurring. At temperature 0.1, only the most likely token has a chance of being selected. At temperature 2.0, the most likely token is still the most likely, but the probabilities of the other tokens have been boosted in comparison; at temperature 50.0, no token is considered to be more likely than any other.

Now let's go back to our original example and play around with the temperature. Try a small value (< 1) and a very large value ( > 10) and see what results.

In [ ]:
# torch.manual_seed(0)   # uncomment for reproducibility
generator = pipeline('text-generation',
                     model = 'openai-community/gpt2',
                     # device = 'cuda'. # set cuda for GPUs; we don't need this yet so let's not
                     )

prompt = "People have a lot of opionions about cilantro. Some people think cilantro tastes like"

responses = generator(prompt,
                      max_new_tokens = 100,
                      num_return_sequences=5,
                      temperature=.1, # <-- HERE IS TEMPERATURE
                      truncation=True
                      )

#  print out the responses
for i, response in enumerate(responses):
    print()
    print(str(i) + ". " + textwrap.fill(response['generated_text'],100))

As you experienced, low temperatures generally produce predictable, repetitive results, while higher temperature example produces less likely sequences of words. The text is a bit livelier, but sometimes at the cost of coherence.

Adjusting the temperature can be useful when you want the text to be more or less "weird." It can be helpful to adjust the temperature downward when you feel as though the model is producing text that is a bit too unpredictable; it can be helpful to adjust upward when you want to model to take more unexpected turns when generating.

It's also helpful to know about (and think about) in the context of this lab, where we will be looking at bias in LLMs, since the *most likely* words, according to the training data, might end up being more stereotypical (or biased).

## Other Transformers Pipelines

The tasks that most people are familiar with are things like text generation and chat. But there are a lot more that are used for NLP research. A big one is called [masked langauge modeling](https://huggingface.co/docs/transformers/en/tasks/masked_language_modeling), which predicts a masked token in a sequence of words.

This is useful when you want to audit a model for some form of bias; or when you're trying to determine the similarity of one word to others; or if you want to identify typos and other word-level errors in your data that you want to fix; or many other use cases. It's a good thing to know about!

For our first MLM exercise, we're going to use the [DistilRoBERTa](https://huggingface.co/distilbert/distilroberta-base) base model. The [RoBERTa](https://huggingface.co/FacebookAI/roberta-base) family of models are trained on English language texts for the specific task of masked langauge modeling, so they're the standard models to use.

The "distiled" version is a smaller, faster, *distilled* version of the larger model. (This terminology is used more broadly in LLM-land). The goal with distilled models is to reduce the number of parameters while preserving performance. It's often true that you can prune a significant number of parameters while still attaining a high degree of performance.  

In any case, this is how you do it:

In [ ]:
# torch.manual_seed(0)   # uncomment for reproducibility
unmasker = pipeline('fill-mask',  # note different pipeline
                     model = 'distilroberta-base',  # note different model
                     # device = 'cuda'. # set cuda for GPUs; we don't need this yet so let's not
                     )

template = "Last weekend, I went to the <mask>"

responses = unmasker(template,
                    top_k = 5, # default is also 5
                      )

responses

What you are looking at here is:

1.  The confidence score
2.  The token ID
3.  The token that's been suggested; and
4.  The completed sequence.

The confidence score becomes a helpful metric if you want to do something like provide quantative evidence of bias, or sum over possible choices, or something like that.

## A mini experiment

Let's put these scores to use in a mini experiment: scanning for gender bias in occupations in the model. You'll recognize some familiar syntax here:

In [ ]:
pronouns = ["He", "She", "They"]

# define our template function
def construct_template (pronoun):
  template = "After college, " + pronoun + " worked as a <mask>."
  return template

# define our unmasker function
def unmask_gender(template):
  responses = unmasker(template, top_k = 5)

  return responses

# put both to use
for pronoun in pronouns:
  responses = unmask_gender(construct_template(pronoun))

  print("Pronoun: " + pronoun)

  for i, response in enumerate(responses):
    print(responses[i]["token_str"] + " " + str(responses[i]["score"]))

  print("\n")

## Masked Language Modeling with Historical Models ##

We've all heard about the problems with LLM training data, and how it mostly comes from the internet. But as it turns out, there are researchers who are taking a different apporach, by training LLMs using only historical data. In fact, that's how the [BLERT](https://huggingface.co/Livingwithmachines/bert_1760_1900) model was trained: using a giant number of digitized newspapers and other texts from the British Library that span the years 1760 to 1900. And now we're going to use it for our MLM task!

Remember in our first experiment, how we asked the model to fill in the sentence, "Last week, I went to the [MASK]", and it came back with things like "beach" and "movies" and "gym"? Now let's see what the historical model does.

In [ ]:
# torch.manual_seed(0)   # uncomment for reproducibility
unmasker = pipeline('fill-mask',  # note different pipeline
                     model = 'Livingwithmachines/bert_1760_1850',  # note different model
                     # device = 'cuda'. # set cuda for GPUs; we don't need this yet so let's not
                     )

template = "Last weekend, I went to the [MASK]"

responses = unmasker(template,
                    top_k = 5, # default is also 5
                      )

responses

The takeaway, here, is that *training data matters.*  Thinking back to Donna Haraway, we might say that training data (or the model) is *situated*. Or we might recall the quote from Ngũgĩ wa Thiong'o's *Decolonizing the Mind*:

> “Language carries culture, and culture carries... the entire body of values by which we come to perceive ourselves and our place in the world. How people perceive themselves affects how they look at their culture, at their places, their politics and at the social production of wealth, at their entire relationship to nature and to other beings. Language is thus inseparable from ourselves as a community of human beings with a specific form and character, a specific history, a specific relationship to the world.”

> — Ngũgĩ wa Thiong'o, Decolonising the Mind: The Politics of Language in African Literature (1986), p. 16.